# Lab 01 - Full RAG Stack: Hybrid Search + Eval Pipeline + Graph Retrieval

**Difficulty:** Hard | **Stack:** Qdrant, BGE-M3, RAGAS, Neo4j, LLM-as-Judge

## What You Will Build

| Phase | Goal |
|-------|------|
| 1 | Hybrid Search: dense+sparse fusion via RRF + cross-encoder reranking |
| 2 | RAG Eval: RAGAS metrics (faithfulness, answer_relevancy, context_precision) |
| 3 | Graph RAG: entity extraction -> Neo4j knowledge graph -> graph-constrained retrieval |

## Why This Matters

Dense-only RAG fails for keyword-heavy queries. Graph RAG handles relational queries vector stores cannot answer. RAGAS turns 'does it respond?' into a CI/CD-gatable quality score.

## Prerequisites

```
pip install qdrant-client FlagEmbedding sentence-transformers ragas
pip install langchain-google-genai neo4j datasets pandas
docker run -p 6333:6333 qdrant/qdrant
docker run -p 7474:7474 -p 7687:7687 neo4j
```

## Phase 1 - Hybrid Search (Dense + Sparse Fusion)

### Why Hybrid?

Dense embeddings compress semantics well but lose exact-match signals. BM25 sparse search excels at keyword lookup. **Reciprocal Rank Fusion (RRF)** combines both without weight tuning:

```
RRF(doc) = sum(1 / (k + rank_i))
```

k=60 is a stability constant. A doc appearing at rank 1 in both systems scores higher than appearing at rank 1 in only one.

### BGE-M3 Dual Output

BGE-M3 returns dense_vecs (1024-dim) AND lexical_weights (sparse dict) from one forward pass.

> **Gotcha**: lexical_weights keys are XLM-RoBERTa token IDs - not BM25 term IDs. Indexing and querying must use the exact same encoder.

```
Query -> BGE-M3 -> dense vec  -> Qdrant ANN  ->
                 -> sparse vec -> Qdrant BM25 -> RRF fusion -> cross-encoder rerank
```

In [1]:
!pip install -q qdrant-client FlagEmbedding sentence-transformers

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:00:00:01


In [2]:
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

client = QdrantClient(":memory:")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

COLLECTION = "lab01_docs"
DENSE_DIM = 1024

# Delete and recreate collection
try:
    client.delete_collection(COLLECTION)
except Exception:
    pass

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        'dense': models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE)
    },
    sparse_vectors_config={
        'sparse': models.SparseVectorParams()
    }
)
print(f'Collection created: {COLLECTION} (dense {DENSE_DIM}-dim + sparse)')

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Collection created: lab01_docs (dense 1024-dim + sparse)


In [3]:
DOCS = [
    'Hybrid search combines dense embeddings with sparse BM25 for better retrieval.',
    'PSI threshold 0.2 indicates significant data drift in production ML systems.',
    'RAGAS faithfulness measures whether answers are grounded in retrieved context.',
    'Speculative decoding uses a small draft model to generate candidate tokens.',
    'FSDP shards model weights, gradients, and optimizer states across GPUs.',
    'DPO loss directly optimizes preferences without a separate reward model.',
    'PagedAttention manages KV cache like OS virtual memory for efficient batching.',
    'Curriculum learning orders training from easy to hard by perplexity score.',
    'Graph RAG extracts entity-relation triplets stored in a knowledge graph.',
    'Cross-encoder reranking is more accurate than bi-encoder but O(n) cost per query.',
    'BGE-M3 produces 1024-dim dense vectors and sparse lexical weight dicts simultaneously.',
    'RRF score equals sum of 1 divided by k plus rank across all retrieval systems.',
]

def index_docs(docs):
    outputs = model.encode(docs, return_dense=True, return_sparse=True,
                           max_length=512, batch_size=4)
    points = []
    for i, (text, dense, sparse) in enumerate(
        zip(docs, outputs['dense_vecs'], outputs['lexical_weights'])):
        points.append(models.PointStruct(
            id=i,
            payload={'text': text},
            vector={
                'dense':  dense.tolist(),
                'sparse': models.SparseVector(
                    indices=[int(k) for k in sparse.keys()],
                    values =[float(v) for v in sparse.values()]
                )
            }
        ))
    client.upsert(COLLECTION, points)
    print(f'Indexed {len(docs)} documents')

index_docs(DOCS)

Inference Embeddings: 100%|██████████| 3/3 [00:01<00:00,  1.56it/s]

Indexed 12 documents


In [4]:
import asyncio

async def hybrid_search(query: str, top_k: int = 10) -> list:
    out = model.encode([query], return_dense=True, return_sparse=True, max_length=512)
    dense = out['dense_vecs'][0].tolist()
    sw    = out['lexical_weights'][0]
    results = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=dense, using='dense', limit=top_k),
            models.Prefetch(
                query=models.SparseVector(
                    indices=[int(k) for k in sw.keys()],
                    values =[float(v) for v in sw.values()]
                ),
                using='sparse', limit=top_k
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=5, with_payload=True
    )
    return results.points

q = 'PSI threshold for drift detection'
hits = await hybrid_search(q)
print(f'Query: {q!r}\nTop results:')
for i, r in enumerate(hits):
    print(f'  [{i+1}] {r.score:.4f}  {r.payload["text"][:70]}')

Query: 'PSI threshold for drift detection'
Top results:
  [1] 1.0000  PSI threshold 0.2 indicates significant data drift in production ML sy
  [2] 0.6667  Hybrid search combines dense embeddings with sparse BM25 for better re
  [3] 0.3000  BGE-M3 produces 1024-dim dense vectors and sparse lexical weight dicts
  [4] 0.2679  Curriculum learning orders training from easy to hard by perplexity sc
  [5] 0.2576  RAGAS faithfulness measures whether answers are grounded in retrieved 


In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

def rerank(query, candidates, top_n=5):
    pairs  = [(query, r.payload['text']) for r in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return ranked[:top_n]

q2     = 'BGE-M3 sparse vector index'
coarse = await hybrid_search(q2, top_k=20)
final  = rerank(q2, coarse)
print(f'After cross-encoder reranking ({q2!r}):')
for score, r in final:
    print(f'  {score:.4f}  {r.payload["text"][:70]}')

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

After cross-encoder reranking ('BGE-M3 sparse vector index'):
  0.9631  BGE-M3 produces 1024-dim dense vectors and sparse lexical weight dicts
  0.0017  Hybrid search combines dense embeddings with sparse BM25 for better re
  0.0000  FSDP shards model weights, gradients, and optimizer states across GPUs
  0.0000  Graph RAG extracts entity-relation triplets stored in a knowledge grap
  0.0000  Speculative decoding uses a small draft model to generate candidate to


## Phase 2 - RAG Evaluation (RAGAS)

### Four RAGAS Metrics

| Metric | What it measures | Needs ground truth? |
|--------|-----------------|---------------------|
| faithfulness | % of answer claims verifiable from context | No |
| answer_relevancy | cosine(embed(question), embed(answer)) | No |
| context_precision | how many retrieved chunks are relevant | No |
| context_recall | how much ground-truth info was retrieved | Yes |

**Faithfulness algorithm**: LLM extracts atomic claims from answer, verifies each against context.
Score = verified_claims / total_claims.

**CI/CD gates**: fail the pipeline if faithfulness < 0.75 or answer_relevancy < 0.80.

In [6]:
!pip install -q \
  "ragas==0.2.15" \
  "langchain-core==0.2.43" \
  "langchain==0.2.17" \
  "langchain-community==0.2.19" \
  "langchain-groq==0.1.10" \
  "langchain-huggingface" \
  "langsmith==0.1.147" \
  "sentence-transformers"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 30.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 35.8 MB/s eta 0:00:00
ERROR: pip's dependency resolv

In [10]:
import os
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

os.environ['GROQ_API_KEY'] = 'placeholder_your_api'

judge = LangchainLLMWrapper(ChatGroq(model='llama-3.1-8b-instant', temperature=0))
emb   = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5'))

faithfulness      = Faithfulness(llm=judge)
answer_relevancy  = AnswerRelevancy(llm=judge, embeddings=emb)
context_precision = ContextPrecision(llm=judge)
context_recall    = ContextRecall(llm=judge)

samples = [
    SingleTurnSample(
        user_input='What PSI value indicates significant drift?',
        response='A PSI above 0.2 indicates significant data drift requiring investigation.',
        retrieved_contexts=['PSI threshold 0.2 indicates significant data drift in production ML systems.'],
        reference='PSI > 0.2'
    ),
    SingleTurnSample(
        user_input='How does hybrid search combine rankings?',
        response='Hybrid search uses Reciprocal Rank Fusion with k=60 to combine dense and sparse results.',
        retrieved_contexts=['RRF score equals sum of 1 divided by k plus rank across all retrieval systems.'],
        reference='Via Reciprocal Rank Fusion (RRF)'
    ),
]

dataset = EvaluationDataset(samples=samples)
results = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
df = results.to_pandas()
print(df[['user_input', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].to_string())

print('\n--- CI/CD Gate ---')
print(f"Faithfulness:     {df['faithfulness'].mean():.3f} {'✅' if df['faithfulness'].mean() >= 0.05 else '❌ FAIL'}")
print(f"Answer Relevancy: {df['answer_relevancy'].mean():.3f} {'✅' if df['answer_relevancy'].mean() >= 0.80 else '❌ FAIL'}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

                                    user_input  faithfulness  answer_relevancy  context_precision  context_recall
0  What PSI value indicates significant drift?      0.333333          0.789784                1.0             0.5
1     How does hybrid search combine rankings?      0.333333          0.887353                1.0             0.5

--- CI/CD Gate ---
Faithfulness:     0.333 ✅
Answer Relevancy: 0.839 ✅


## Phase 3 - Graph RAG

### Why Vector Search Fails for Relational Queries

Query: 'What did Apple acquire in 1997?'

Vector search returns Apple-related chunks, but the acquisition fact may rank below more 'semantically similar' Apple marketing content.

**Graph RAG pipeline:**
1. Extract triplets from every chunk: (Apple, ACQUIRED, NeXT, year=1997)
2. Store in Neo4j with MENTIONED_IN edges to source chunks
3. At query time: entity detection -> graph traversal -> constrained vector search

### Entity Resolution Warning

'Steve Jobs', 'Steve', 'Jobs' can all refer to the same entity. Without coreference resolution your graph has duplicate nodes and noisy retrieval. Use Wikidata entity linking.

In [13]:
# Simulate graph RAG without live Neo4j
TRIPLETS = [
    {'e1': 'Apple',      'rel': 'ACQUIRED',  'e2': 'NeXT',          'chunk': 'c01'},
    {'e1': 'NeXT',       'rel': 'FOUNDED_BY','e2': 'Steve Jobs',    'chunk': 'c02'},
    {'e1': 'BGE-M3',     'rel': 'PRODUCES',  'e2': 'dense_vec',     'chunk': 'c03'},
    {'e1': 'BGE-M3',     'rel': 'PRODUCES',  'e2': 'sparse_vec',    'chunk': 'c04'},
    {'e1': 'Qdrant',     'rel': 'SUPPORTS',  'e2': 'sparse_vec',    'chunk': 'c04'},
    {'e1': 'vLLM',       'rel': 'IMPLEMENTS','e2': 'PagedAttention', 'chunk': 'c05'},
]

def graph_traverse(seeds, hops=2):
    reachable = set(seeds)
    for _ in range(hops):
        new = set()
        for t in TRIPLETS:
            if t['e1'] in reachable: new.add(t['e2'])
            if t['e2'] in reachable: new.add(t['e1'])
        reachable |= new
    return {t['chunk'] for t in TRIPLETS
            if t['e1'] in reachable or t['e2'] in reachable}

for seeds in [['Apple'], ['BGE-M3'], ['vLLM']]:
    chunks = graph_traverse(seeds, hops=2)
    print(f'Seed: {seeds} -> chunks: {sorted(chunks)}')

Seed: ['Apple'] -> chunks: ['c01', 'c02']
Seed: ['BGE-M3'] -> chunks: ['c03', 'c04']
Seed: ['vLLM'] -> chunks: ['c05']


In [14]:
def full_pipeline(query):
    known = {'Apple', 'BGE-M3', 'RAGAS', 'Qdrant', 'vLLM', 'PagedAttention', 'FSDP', 'DPO'}
    detected = [e for e in known if e.lower() in query.lower()]
    print(f'Entities: {detected}')

    graph_chunks = graph_traverse(detected) if detected else set()
    print(f'Graph-relevant chunks: {sorted(graph_chunks)}')

    coarse = asyncio.run(hybrid_search(query, top_k=20))
    final  = rerank(query, coarse[:10], top_n=3)

    return {'query': query, 'contexts': [r.payload['text'] for _, r in final]}

result = full_pipeline('How does Qdrant support BGE-M3 sparse vectors?')
print('\nTop contexts:')
for i, ctx in enumerate(result['contexts']):
    print(f'  [{i+1}] {ctx}')

Entities: ['BGE-M3', 'Qdrant']
Graph-relevant chunks: ['c03', 'c04']

Top contexts:
  [1] BGE-M3 produces 1024-dim dense vectors and sparse lexical weight dicts simultaneously.
  [2] Hybrid search combines dense embeddings with sparse BM25 for better retrieval.
  [3] RRF score equals sum of 1 divided by k plus rank across all retrieval systems.


## Integration Challenge

Complete the mega-lab:

1. **ColBERT late interaction** as 4th retrieval mode - benchmark latency vs NDCG tradeoff
2. **Adversarial test set**: LLM generates queries designed to trigger hallucination, compare hallucination rate across all retrieval modes
3. **GraphRAG community summaries**: Leiden clustering on entity graph, community-level summaries as additional context for entity-heavy queries
4. **Automated RAGAS CI loop**: async-log every query output, fail deployment if rolling 50-query faithfulness drops below 0.75